# 전체 노선 Ridge 잔여좌석 모델 분석 및 시각화

현재 `data/csv`의 전체 노선 데이터로 학습한 Ridge 모델의 결과를 분석합니다. Random Forest는 사용하지 않습니다.


## 0. 최신 데이터로 모델 다시 만들기

데이터가 갱신됐다면 먼저 학습 스크립트를 실행합니다. 현재 터미널 위치에 따라 둘 중 하나를 사용합니다.

프로젝트 루트(`10th-toy-team4`)에서 실행할 때:

```bash
.venv/bin/python sanghyuk/analyze_and_train.py
```

`sanghyuk` 폴더에서 실행할 때:

```bash
../.venv/bin/python analyze_and_train.py
```

노트북의 아래 설정 셀은 프로젝트 루트와 `sanghyuk` 중 어디에서 실행해도 데이터 경로를 자동으로 찾습니다.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root():
    """Jupyter 실행 위치와 관계없이 10th-toy-team4 루트를 찾는다."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / "10th-toy-team4"]
    for candidate in candidates:
        if (candidate / "data/csv/history_all.csv").is_file() and (candidate / "sanghyuk").is_dir():
            return candidate
    checked = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. history_all.csv가 있는지 확인하세요.\n"
        f"확인한 위치:\n{checked}"
    )

PROJECT_ROOT = find_project_root()
VISUALIZATION_DIR = PROJECT_ROOT / "sanghyuk/visualizations"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = PROJECT_ROOT / "data/analysis/models"
REPORT_PATH = MODEL_DIR / "model_report.json"
PREDICTION_PATH = MODEL_DIR / "predictions.csv"
HISTORY_PATH = PROJECT_ROOT / "data/csv/history_all.csv"
ROUTE_NAMES_PATH = PROJECT_ROOT / "sanghyuk/route_names.csv"
print("project root:", PROJECT_ROOT)
print("report path:", REPORT_PATH)
print("visualization output:", VISUALIZATION_DIR)

def save_visualization(fig, filename):
    """화면에 표시하는 Figure를 GitHub 가이드용 PNG로 함께 저장한다."""
    path = VISUALIZATION_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"PNG updated: {path}")

def create_model_outputs():
    """보고서가 없거나 손상됐을 때 Ridge 학습 스크립트로 다시 생성한다."""
    script_path = PROJECT_ROOT / "sanghyuk/analyze_and_train.py"
    print(f"모델 보고서를 생성합니다: {script_path}")
    subprocess.run(
        [sys.executable, str(script_path)],
        cwd=PROJECT_ROOT,
        check=True,
    )

try:
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
except (FileNotFoundError, json.JSONDecodeError) as error:
    print(f"보고서를 읽지 못했습니다: {error}")
    create_model_outputs()
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

if not PREDICTION_PATH.is_file():
    create_model_outputs()

predictions = pd.read_csv(PREDICTION_PATH)
predictions["observed_at"] = pd.to_datetime(predictions["observed_at"], utc=True)
predictions["observed_at_kst"] = predictions["observed_at"].dt.tz_convert("Asia/Seoul")
predictions["hour"] = predictions["observed_at_kst"].dt.hour
predictions["error"] = predictions["arrival_remaining_seats"] - predictions["ridge_prediction"]
predictions["absolute_error"] = predictions["error"].abs()

# 노선별 현황은 최근 테스트 구간이 아니라 전체 수집 이력을 사용한다.
history_columns = pd.read_csv(HISTORY_PATH, nrows=0).columns
route_columns = ["observed_at", "route_id", "remaining_seats"]
if "route_name" in history_columns:
    route_columns.append("route_name")
route_history = pd.read_csv(HISTORY_PATH, usecols=route_columns, dtype={"route_id": str})
route_history["observed_at"] = pd.to_datetime(route_history["observed_at"], errors="coerce", utc=True)
route_history["remaining_seats"] = pd.to_numeric(route_history["remaining_seats"], errors="coerce")
route_history = route_history.dropna(subset=["observed_at", "route_id", "remaining_seats"])
route_history = route_history[route_history["remaining_seats"] >= 0].copy()
route_history["hour"] = route_history["observed_at"].dt.tz_convert("Asia/Seoul").dt.hour
route_history["actual_low_seat"] = route_history["remaining_seats"].le(10).astype(int)

# 새 history_all.csv는 route_name을 직접 포함한다. 기존 파일이면 routes.csv에서 보완한다.
routes = pd.read_csv(ROUTE_NAMES_PATH, dtype={"route_id": str})
route_history["route_id"] = route_history["route_id"].astype(str)
if "route_name" not in route_history.columns:
    route_history["route_name"] = pd.NA
if "route_name" in routes.columns:
    route_names = (
        routes.dropna(subset=["route_name"])
        .drop_duplicates("route_id")
        .set_index("route_id")["route_name"]
        .astype(str)
    )
    route_history["route_name"] = route_history["route_name"].fillna(
        route_history["route_id"].map(route_names)
    )
missing_route_ids = sorted(
    route_history.loc[route_history["route_name"].isna(), "route_id"].unique()
)
def require_route_names():
    """노선명 준비 여부를 반환하고 필요한 실행 명령을 안내한다."""
    if not missing_route_ids:
        return True
    print(
        "노선별 시각화를 건너뜁니다: route_name 매핑이 비어 있습니다.\n"
        "1) 프로젝트 .env에 GBIS_SERVICE_KEY를 설정하고\n"
        "2) `.venv/bin/python sanghyuk/fetch_route_names.py`를 실행한 뒤\n"
        "3) 이 노트북의 커널을 재시작하세요.\n"
        f"누락 노선: {len(missing_route_ids)}개"
    )
    return False

if missing_route_ids:
    print(f"route_name 동기화 필요: {len(missing_route_ids)}개 노선")
route_history["route_label"] = route_history["route_name"].astype("string")

print("rows:", report["rows"])
print("split:", report["split"])
print("all collected routes for route charts:", route_history["route_id"].nunique())
predictions.head()


## 1. 회귀 및 분류 평가표


In [ ]:
display(pd.DataFrame(report["metrics"]).T)
display(pd.DataFrame(report["within_10_test"]["metrics"]).T)
display(pd.DataFrame(report["low_seat_classification"]["metrics"]).T)


## 2. 실제값과 Ridge 예측값 산점도

점이 점선에 가까울수록 예측이 정확합니다. 데이터가 많아 최대 20,000건을 고정된 난수로 표본 추출합니다.


In [ ]:
sample = predictions.sample(min(20_000, len(predictions)), random_state=42)
limit = max(sample["arrival_remaining_seats"].max(), sample["ridge_prediction"].max())
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(sample["arrival_remaining_seats"], sample["ridge_prediction"], s=8, alpha=0.2)
ax.plot([0, limit], [0, limit], "r--", linewidth=1.5, label="Perfect prediction")
ax.set(xlabel="Actual remaining seats", ylabel="Ridge prediction", title="Actual vs Ridge prediction")
ax.legend()
ax.grid(alpha=0.2)
save_visualization(fig, "01_actual_vs_prediction.png")
plt.show()


## 3. 실제 잔여좌석 구간별 MAE

저잔여석 구간과 일반 구간 중 어디에서 오차가 커지는지 비교합니다.


In [ ]:
seat_bins = [-0.001, 5, 10, 20, np.inf]
seat_labels = ["0-5", "6-10", "11-20", "21+"]
predictions["seat_range"] = pd.cut(
    predictions["arrival_remaining_seats"], bins=seat_bins, labels=seat_labels
)
range_mae = predictions.groupby("seat_range", observed=False)["absolute_error"].agg(["mean", "count"])
range_mae.columns = ["MAE", "Rows"]
display(range_mae)
ax = range_mae["MAE"].plot(kind="bar", figsize=(8, 4), color="#4C78A8", rot=0)
ax.set(title="MAE by actual remaining-seat range", xlabel="Actual seat range", ylabel="MAE")
ax.grid(axis="y", alpha=0.2)
save_visualization(ax.figure, "02_mae_by_seat_range.png")
plt.show()


## 4. 만석 경고 임계값별 Precision·Recall·F1-score

실제 잔여좌석 0석을 양성으로 두고, Ridge 예측값이 임계값보다 작으면 만석 경고를 발생시킵니다.


In [ ]:
threshold_metrics = pd.DataFrame(
    report["full_bus_threshold_sweep"]["thresholds"]
).T.astype(float)
threshold_metrics.index = threshold_metrics.index.astype(float)
score_columns = ["Precision", "Recall", "F1-score"]
ax = (threshold_metrics[score_columns] * 100).plot(marker="o", figsize=(8, 5))
ax.set(
    title="Full-bus warning scores by threshold",
    xlabel="Warning threshold: prediction < threshold",
    ylabel="Score (%)",
    xticks=threshold_metrics.index,
    ylim=(0, 100),
)
ax.grid(alpha=0.25)
save_visualization(ax.figure, "03_threshold_scores.png")
plt.show()
(threshold_metrics[score_columns] * 100).round(4)


## 5. 만석 경고 임계값별 FP·FN

FP는 실제로 만석이 아닌데 경고한 건수이고, FN은 실제 만석을 놓친 건수입니다.


In [ ]:
error_counts = threshold_metrics[["False positive", "False negative"]].rename(
    columns={"False positive": "FP (false alarm)", "False negative": "FN (missed full bus)"}
)
ax = error_counts.plot(kind="bar", figsize=(9, 5), rot=0, color=["#F58518", "#E45756"])
ax.set(title="False alarms and missed full buses", xlabel="Warning threshold", ylabel="Cases")
ax.grid(axis="y", alpha=0.2)
save_visualization(ax.figure, "04_threshold_fp_fn.png")
plt.show()
error_counts.astype(int)


## 6. 실제값과 예측값의 시간 흐름

테스트셋 마지막 500개 관측을 시간순으로 표시합니다.


In [ ]:
timeline = predictions.sort_values("observed_at_kst").tail(min(500, len(predictions)))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(timeline["observed_at_kst"], timeline["arrival_remaining_seats"], label="Actual", linewidth=1.2)
ax.plot(timeline["observed_at_kst"], timeline["ridge_prediction"], label="Ridge", linewidth=1)
ax.set(title="Actual and predicted seats over time", xlabel="Observed time (KST)", ylabel="Seats")
ax.legend()
ax.grid(alpha=0.2)
fig.autofmt_xdate()
save_visualization(fig, "05_prediction_timeline.png")
plt.show()


## 7. 예측 오차 분포

오차는 `실제값 - 예측값`입니다. 양수면 모델이 실제보다 적게 예측했고, 음수면 실제보다 많이 예측했습니다.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(predictions["error"], bins=80, color="#4C78A8", alpha=0.85)
ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
ax.set(title="Prediction error distribution", xlabel="Actual - prediction", ylabel="Rows")
ax.grid(axis="y", alpha=0.2)
save_visualization(fig, "06_error_distribution.png")
plt.show()
predictions["error"].describe()


## 8. 시간대별 MAE

출퇴근 시간 등 특정 시간대에서 오차가 커지는지 확인합니다.


In [ ]:
hourly_mae = predictions.groupby("hour")["absolute_error"].agg(["mean", "count"])
hourly_mae.columns = ["MAE", "Rows"]
ax = hourly_mae["MAE"].plot(marker="o", figsize=(10, 4), xticks=range(24))
ax.set(title="MAE by hour", xlabel="Hour (KST)", ylabel="MAE")
ax.grid(alpha=0.25)
save_visualization(ax.figure, "07_hourly_mae.png")
plt.show()
hourly_mae


## 9. 노선별 시간대 저잔여석 건수

저잔여석은 실제 잔여좌석이 10석 이하인 경우입니다. 모든 노선을 포함하기 위해 모델 테스트셋이 아니라 전체 수집 이력 `history_all.csv`를 사용하고, 노선명은 로컬 `sanghyuk/route_names.csv`에서 매핑합니다. 각 행은 노선, 각 열은 한국 시간대이며 색이 진할수록 저잔여석 관측 건수가 많습니다.


In [ ]:
if require_route_names():
    route_hour = route_history.groupby(["route_label", "hour"])["actual_low_seat"].agg(["sum", "count"])
    low_count = route_hour["sum"].unstack(fill_value=0).reindex(columns=range(24), fill_value=0)

    fig_height = max(5, 0.42 * len(low_count))
    fig, ax = plt.subplots(figsize=(14, fig_height))
    image = ax.imshow(low_count.to_numpy(), aspect="auto", cmap="YlOrRd")
    ax.set(
        title="Low-seat observations by route and hour (actual seats <= 10)",
        xlabel="Hour (KST)", ylabel="Route name",
        xticks=np.arange(24), yticks=np.arange(len(low_count)),
        yticklabels=low_count.index.astype(str),
    )
    fig.colorbar(image, ax=ax, label="Low-seat rows")
    fig.tight_layout()
    save_visualization(fig, "08_route_hour_low_seat_count.png")
    plt.show()
    low_count


## 10. 노선별 시간대 저잔여율

저잔여율은 각 노선·시간대 전체 관측 중 실제 잔여좌석이 10석 이하인 비율입니다. 단순 건수와 달리 시간대별 관측량 차이를 보정합니다. 관측이 없는 조합은 빈칸으로 표시합니다.


In [ ]:
if require_route_names():
    low_rate = (route_hour["sum"] / route_hour["count"] * 100).unstack().reindex(columns=range(24))
    masked_rate = np.ma.masked_invalid(low_rate.to_numpy(dtype=float))

    fig_height = max(5, 0.42 * len(low_rate))
    fig, ax = plt.subplots(figsize=(14, fig_height))
    image = ax.imshow(masked_rate, aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)
    ax.set(
        title="Low-seat rate by route and hour (actual seats <= 10)",
        xlabel="Hour (KST)", ylabel="Route name",
        xticks=np.arange(24), yticks=np.arange(len(low_rate)),
        yticklabels=low_rate.index.astype(str),
    )
    fig.colorbar(image, ax=ax, label="Low-seat rate (%)")
    fig.tight_layout()
    save_visualization(fig, "09_route_hour_low_seat_rate.png")
    plt.show()
    low_rate.round(2)


## 11. 노선별 저잔여율 상세 선 그래프

각 노선의 시간대별 저잔여율을 개별 서브플롯으로 확인합니다. 관측이 없는 시간대는 선이 연결되지 않습니다.


In [ ]:
if require_route_names():
    route_labels = list(low_rate.index)
    cols = 3
    rows = int(np.ceil(len(route_labels) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(15, max(4, rows * 3)), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, route_label in zip(axes, route_labels):
        values = low_rate.loc[route_label]
        ax.plot(values.index, values.values, marker="o", markersize=3, linewidth=1.2)
        ax.set_title(str(route_label))
        ax.set_xticks(range(0, 24, 3))
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.2)
    for ax in axes[len(route_labels):]:
        ax.set_visible(False)
    fig.supxlabel("Hour (KST)")
    fig.supylabel("Low-seat rate (%)")
    fig.suptitle("Hourly low-seat rate for each route", y=1.01)
    fig.tight_layout()
    save_visualization(fig, "10_route_hour_low_seat_lines.png")
    plt.show()


## 12. 저장된 Ridge 모델 불러오기


In [ ]:
ridge_model = joblib.load(MODEL_DIR / "ridge_model.joblib")
ridge_model
